# Analisis Penjualan, Regresi Revenue, dan Break-Even Point (BEP)

Notebook ini mengerjakan:
1. Plot jumlah penjualan vs revenue untuk kelima gerai.
2. Regresi untuk melihat bentuk tren revenue dari volume penjualan.
3. Mencari volume penjualan yang menghasilkan Break-Even Point (BEP).

In [ ]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Persiapan Data

In [ ]:
DATA_PATH = Path("sintesis_data_donat_harian.csv")

with open(DATA_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Jumlah data: {len(rows)}")
print("Kolom:", reader.fieldnames)

In [ ]:
df = pd.DataFrame(rows)

numeric_cols = [
    "Harga Jual (Rp)",
    "Jumlah Terjual (Unit)",
    "Pendapatan Kotor (Rp)",
    "Biaya Variabel (Rp)",
    "Fixed Cost Harian (Rp)",
    "Balance (Rp)"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("Rp", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[
    "Gerai",
    "Jumlah Terjual (Unit)",
    "Pendapatan Kotor (Rp)",
    "Biaya Variabel (Rp)",
    "Fixed Cost Harian (Rp)"
])

df.head()

## 1. Plot Jumlah Penjualan vs Revenue

In [ ]:
stores = sorted(df["Gerai"].unique())

print("Gerai yang dianalisis:")
for store in stores:
    print("-", store)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, store in enumerate(stores):
    data_store = df[df["Gerai"] == store]
    x = data_store["Jumlah Terjual (Unit)"]
    y = data_store["Pendapatan Kotor (Rp)"]

    axes[i].scatter(x, y, alpha=0.7)
    axes[i].set_title(f"{store}: Jumlah Penjualan vs Revenue")
    axes[i].set_xlabel("Jumlah Terjual (Unit)")
    axes[i].set_ylabel("Revenue (Rp)")
    axes[i].grid(alpha=0.3)

for j in range(len(stores), len(axes)):
    axes[j].axis("off")

plt.suptitle("Jumlah Penjualan vs Revenue untuk Kelima Gerai", fontsize=16)
plt.tight_layout()
plt.show()

### Visualisasi Gabungan Kelima Gerai

In [ ]:
plt.figure(figsize=(12, 7))

for store in stores:
    data_store = df[df["Gerai"] == store]
    plt.scatter(
        data_store["Jumlah Terjual (Unit)"],
        data_store["Pendapatan Kotor (Rp)"],
        label=store,
        alpha=0.7
    )

plt.xlabel("Jumlah Terjual (Unit)")
plt.ylabel("Revenue (Rp)")
plt.title("Jumlah Penjualan vs Revenue - Kelima Gerai")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Regresi Revenue terhadap Volume Penjualan

Model yang dibandingkan:
- Linear
- Polinomial orde 2
- Eksponensial

Model dengan nilai R² terbesar dipilih sebagai model terbaik.

In [ ]:
def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return 1.0

    return 1 - (ss_res / ss_tot)

In [ ]:
def fit_revenue_models(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    models = []

    # Linear
    coef_linear = np.polyfit(x, y, 1)
    pred_linear = np.polyval(coef_linear, x)
    r2_linear = r2_score(y, pred_linear)

    models.append({
        "name": "Linear",
        "coef": coef_linear,
        "r2": r2_linear,
        "predict": lambda q, c=coef_linear: np.polyval(c, q)
    })

    # Polinomial orde 2
    coef_poly = np.polyfit(x, y, 2)
    pred_poly = np.polyval(coef_poly, x)
    r2_poly = r2_score(y, pred_poly)

    models.append({
        "name": "Polinomial orde 2",
        "coef": coef_poly,
        "r2": r2_poly,
        "predict": lambda q, c=coef_poly: np.polyval(c, q)
    })

    # Eksponensial
    if np.all(y > 0):
        coef_exp = np.polyfit(x, np.log(y), 1)
        pred_exp = np.exp(coef_exp[1] + coef_exp[0] * x)
        r2_exp = r2_score(y, pred_exp)

        models.append({
            "name": "Eksponensial",
            "coef": coef_exp,
            "r2": r2_exp,
            "predict": lambda q, c=coef_exp: np.exp(c[1] + c[0] * np.asarray(q))
        })

    # Jika R² sangat mirip, prioritaskan model yang lebih sederhana
    priority = {"Linear": 0, "Polinomial orde 2": 1, "Eksponensial": 2}
    max_r2 = max(m["r2"] for m in models)
    tolerance = 1e-10
    candidates = [m for m in models if abs(m["r2"] - max_r2) <= tolerance]
    best_model = min(candidates, key=lambda m: priority[m["name"]])

    return models, best_model

In [ ]:
def get_equation(model):
    name = model["name"]
    coef = model["coef"]

    if name == "Linear":
        return f"Revenue = {coef[0]:,.2f}q + {coef[1]:,.2f}"

    elif name == "Polinomial orde 2":
        return (
            f"Revenue = {coef[0]:,.4f}q² "
            f"+ {coef[1]:,.2f}q "
            f"+ {coef[2]:,.2f}"
        )

    elif name == "Eksponensial":
        return (
            f"Revenue = {np.exp(coef[1]):,.2f} "
            f"× exp({coef[0]:.6f}q)"
        )

    return "Model tidak dikenali" 

In [ ]:
revenue_models = {}
regression_results = []

for store in stores:
    data_store = df[df["Gerai"] == store]
    x = data_store["Jumlah Terjual (Unit)"].to_numpy()
    y = data_store["Pendapatan Kotor (Rp)"].to_numpy()

    models, best_model = fit_revenue_models(x, y)
    revenue_models[store] = best_model

    print("=" * 70)
    print(store)
    print("\nPerbandingan model:")

    for model in models:
        print(f"{model['name']:<22} R² = {model['r2']:.6f}")

    print(f"\nModel terpilih: {best_model['name']}")
    print("Persamaan      :", get_equation(best_model))

    regression_results.append({
        "Gerai": store,
        "Model Terpilih": best_model["name"],
        "R²": best_model["r2"],
        "Persamaan": get_equation(best_model)
    })

regression_df = pd.DataFrame(regression_results)
regression_df

### Visualisasi Hasil Regresi

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, store in enumerate(stores):
    data_store = df[df["Gerai"] == store]
    x = data_store["Jumlah Terjual (Unit)"].to_numpy()
    y = data_store["Pendapatan Kotor (Rp)"].to_numpy()

    model = revenue_models[store]
    q_line = np.linspace(x.min(), x.max(), 300)
    y_line = model["predict"](q_line)

    axes[i].scatter(x, y, alpha=0.7, label="Data Aktual")
    axes[i].plot(q_line, y_line, linewidth=2, label=f"Regresi {model['name']}")
    axes[i].set_title(f"{store}\n{model['name']}, R² = {model['r2']:.4f}")
    axes[i].set_xlabel("Jumlah Terjual (Unit)")
    axes[i].set_ylabel("Revenue (Rp)")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

for j in range(len(stores), len(axes)):
    axes[j].axis("off")

plt.suptitle("Regresi Revenue Berdasarkan Volume Penjualan", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Break-Even Point (BEP)

BEP terjadi ketika **Revenue = Total Cost**.

Total Cost dihitung sebagai:

**Total Cost = Biaya Variabel + Fixed Cost**

In [ ]:
cost_models = {}

for store in stores:
    data_store = df[df["Gerai"] == store].copy()
    x = data_store["Jumlah Terjual (Unit)"].to_numpy()

    total_cost = (
        data_store["Biaya Variabel (Rp)"].to_numpy()
        + data_store["Fixed Cost Harian (Rp)"].to_numpy()
    )

    coef_cost = np.polyfit(x, total_cost, 1)
    pred_cost = np.polyval(coef_cost, x)
    r2_cost = r2_score(total_cost, pred_cost)

    cost_models[store] = {
        "coef": coef_cost,
        "r2": r2_cost,
        "predict": lambda q, c=coef_cost: np.polyval(c, q)
    }

    print("=" * 70)
    print(store)
    print(f"Total Cost = {coef_cost[0]:,.2f}q + {coef_cost[1]:,.2f}")
    print(f"R² Total Cost = {r2_cost:.6f}")

In [ ]:
def find_bep(revenue_predict, cost_predict, q_min=0, q_max=10000, steps=10001):
    q_values = np.linspace(q_min, q_max, steps)
    differences = revenue_predict(q_values) - cost_predict(q_values)

    for i in range(len(q_values) - 1):
        d1 = differences[i]
        d2 = differences[i + 1]

        if d1 == 0:
            return q_values[i]

        if d1 * d2 < 0:
            left = q_values[i]
            right = q_values[i + 1]

            for _ in range(100):
                mid = (left + right) / 2
                d_mid = revenue_predict(mid) - cost_predict(mid)

                if abs(d_mid) < 1e-6:
                    return mid

                d_left = revenue_predict(left) - cost_predict(left)

                if d_left * d_mid <= 0:
                    right = mid
                else:
                    left = mid

            return (left + right) / 2

    return None

In [ ]:
bep_results = []

for store in stores:
    revenue_model = revenue_models[store]
    cost_model = cost_models[store]

    revenue_predict = revenue_model["predict"]
    cost_predict = cost_model["predict"]

    bep = find_bep(revenue_predict, cost_predict)

    if bep is not None:
        minimum_units = int(np.ceil(bep))
        revenue_at_bep = float(revenue_predict(bep))
        cost_at_bep = float(cost_predict(bep))
    else:
        minimum_units = None
        revenue_at_bep = None
        cost_at_bep = None

    bep_results.append({
        "Gerai": store,
        "Model Revenue": revenue_model["name"],
        "BEP (Unit)": bep,
        "Minimum Unit": minimum_units,
        "Revenue pada BEP (Rp)": revenue_at_bep,
        "Total Cost pada BEP (Rp)": cost_at_bep
    })

bep_df = pd.DataFrame(bep_results)
bep_df

### Hasil BEP

In [ ]:
print("HASIL BREAK-EVEN POINT (BEP)")
print("=" * 70)

for _, row in bep_df.iterrows():
    print(f"\n{row['Gerai']}")
    print(f"Model Revenue       : {row['Model Revenue']}")
    print(f"BEP                 : {row['BEP (Unit)']:.2f} unit")
    print(f"Minimum Penjualan   : {int(row['Minimum Unit'])} unit/hari")
    print(f"Revenue pada BEP    : Rp {row['Revenue pada BEP (Rp)']:,.0f}")
    print(f"Total Cost pada BEP : Rp {row['Total Cost pada BEP (Rp)']:,.0f}")

### Visualisasi BEP Setiap Gerai

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, store in enumerate(stores):
    data_store = df[df["Gerai"] == store]
    x = data_store["Jumlah Terjual (Unit)"].to_numpy()
    y = data_store["Pendapatan Kotor (Rp)"].to_numpy()

    revenue_model = revenue_models[store]
    cost_model = cost_models[store]

    bep_row = bep_df[bep_df["Gerai"] == store].iloc[0]
    bep = bep_row["BEP (Unit)"]
    minimum_unit = bep_row["Minimum Unit"]

    q_max = max(x.max() * 1.2, minimum_unit * 1.2)
    q_line = np.linspace(0, q_max, 300)

    revenue_line = revenue_model["predict"](q_line)
    cost_line = cost_model["predict"](q_line)
    bep_revenue = revenue_model["predict"](bep)

    axes[i].scatter(x, y, alpha=0.6, label="Revenue Aktual")
    axes[i].plot(q_line, revenue_line, linewidth=2, label="Revenue Regresi")
    axes[i].plot(q_line, cost_line, linewidth=2, label="Total Cost")
    axes[i].axvline(bep, linestyle="--", linewidth=2, label=f"BEP = {bep:.2f} unit")
    axes[i].scatter([bep], [bep_revenue], s=80, zorder=5)

    axes[i].set_title(f"{store}\nBEP = {bep:.2f} unit")
    axes[i].set_xlabel("Jumlah Terjual (Unit)")
    axes[i].set_ylabel("Rp")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

for j in range(len(stores), len(axes)):
    axes[j].axis("off")

plt.suptitle("Break-Even Point (BEP) Kelima Gerai", fontsize=16)
plt.tight_layout()
plt.show()

## Ringkasan Hasil Analisis

In [ ]:
print("=" * 80)
print("RINGKASAN HASIL ANALISIS")
print("=" * 80)

for store in stores:
    revenue_model = revenue_models[store]
    bep_row = bep_df[bep_df["Gerai"] == store].iloc[0]

    print(f"\n{store}")
    print("-" * 40)
    print(f"Model Revenue     : {revenue_model['name']}")
    print(f"R²                : {revenue_model['r2']:.6f}")
    print(f"Persamaan         : {get_equation(revenue_model)}")
    print(f"BEP               : {bep_row['BEP (Unit)']:.2f} unit")
    print(f"Minimum Penjualan : {int(bep_row['Minimum Unit'])} unit/hari")

## Kesimpulan

1. Hubungan jumlah penjualan dan revenue telah divisualisasikan untuk kelima gerai.
2. Model linear, polinomial orde 2, dan eksponensial dibandingkan berdasarkan nilai R².
3. Model dengan nilai R² terbesar digunakan untuk menggambarkan tren revenue.
4. BEP diperoleh ketika revenue hasil regresi sama dengan total cost hasil regresi.
5. Nilai BEP dibulatkan ke atas untuk memperoleh jumlah minimum unit yang harus terjual agar mencapai titik balik modal.